# Launcher only

All logic lives in `scripts/`. This notebook exists to run four commands. Nothing to re-paste
when the code changes: `!git -C {REPO} pull` in cell 2 is enough.

| cell | does | runtime |
|---|---|---|
| 1 | pip pins | GPU or CPU |
| 2 | mount Drive, pull repo, show VRAM | any |
| 3 | `sweep.py --dry-run` (plan + ETA, no model) | any |
| 4 | `sweep.py --smoke` (1 page x 3 modes) | L4 |
| 5 | `sweep.py --papers-per-bucket N --limit M` | L4 |
| 6 | `score.py` + `plot.py` | CPU |

In [ ]:
# 1. Environment (only needed on a fresh runtime)
# The pip conflict about diffusers/gradio wanting a newer huggingface-hub is expected:
# transformers==4.46.3 pins hub<1.0 and nothing here imports those two.
!pip install -q "transformers==4.46.3" "tokenizers<0.21" "huggingface-hub<1.0" accelerate addict easydict rapidfuzz pandas matplotlib

In [ ]:
# 2. Paths. Change WORK only if your Drive layout differs.
from google.colab import drive
drive.mount("/content/drive")

import os, pathlib, subprocess
WORK = pathlib.Path("/content/drive/MyDrive/ocr-layout-budget")
REPO = pathlib.Path("/content/ocr-layout-token-budget")
URL = "https://github.com/hluaguo/ocr-layout-token-budget"

if REPO.exists():
    print(subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"],
                         text=True, capture_output=True).stdout[-300:])
else:
    print(subprocess.run(["git", "clone", "-q", URL, str(REPO)],
                         text=True, capture_output=True).stderr)

DATA = WORK / "data"
if not (DATA / "pages.jsonl").exists():
    print("untarring", WORK / "data.tar.gz")
    subprocess.run(["tar", "--warning=no-unknown-keyword", "-xzf",
                    str(WORK / "data.tar.gz"), "-C", str(WORK)], check=False)

print("data:", (DATA / "pages.jsonl").exists(),
      "| images:", len(list((DATA / "images").glob("*.png"))),
      "| gold:", len(list((DATA / "paper_gold").glob("*.tex"))))
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader

In [ ]:
# 3. Plan + ETA. No model load, no GPU, safe to run on a CPU runtime.
#    Shows what would be queued, what is already done, and the projected hours.
!python {REPO}/scripts/sweep.py --work {WORK} --papers-per-bucket 2 --dry-run

In [ ]:
# 4. Smoke: one page across all three modes (~5 min incl. model load).
#    Expect 144tok n=0 / 256tok n=0 / 544-1120tok n=2..6, all with pred_chars > 0.
!python {REPO}/scripts/sweep.py --work {WORK} --smoke --hook

In [ ]:
# 5. Real run. Resumable: re-run this cell as often as you like.
#    --papers-per-bucket 2  -> 6 papers / 49 pages / 147 generations
#    --limit 120            -> stop this session after 120 generations
#    --max-new-tokens 4096  -> only if generations hit the 8192 cap
!python {REPO}/scripts/sweep.py --work {WORK} --papers-per-bucket 2 --limit 120

In [ ]:
# 6. Score + plot. Switch to a CPU runtime first (pdflatex on a GPU runtime wastes units).
#    Without texlive the compile metric comes out null; the NED panel still works.
!command -v pdflatex >/dev/null || (apt-get update -qq && apt-get install -y -qq texlive-latex-extra)
!python {REPO}/scripts/score.py --runs {WORK}/runs.csv --pages {DATA}/pages.jsonl \
    --group-by paper --out {WORK}/scores.csv --jobs 8
!python {REPO}/scripts/plot.py --work {WORK}